# Mas alla de la Amnesia: LSTMs y GRUs en PyTorch

Estimados ingenieros, es un placer saludarlos. Como hemos discutido en teoria y en seminarios, la RNN Vanilla es elegante matematicamente, pero sufre una patologia severa: la amnesia ante dependencias de largo plazo debido al desvanecimiento del gradiente.

En este tutorial seguiremos un enfoque ABP (Aprendizaje Basado en Problemas), combinando formalismo matematico con implementacion en PyTorch.

## Hoja de Ruta

1. Fundamento matematico: por que hacen falta compuertas.
2. Implementacion de bloques `nn.LSTM` y `nn.GRU`.
3. Problema aplicado I: auditor de reportes financieros (many-to-one).
4. Problema aplicado II: pronostico de demanda energetica (many-to-many).
5. Tip del doctor: inicializacion del sesgo de olvido y gradient clipping.
6. Retos de investigacion para el estudiante.

In [ ]:
# En Colab normalmente ya esta instalado torch.
# Descomenta si necesitas reinstalar versiones especificas:
# !pip install -q torch torchvision torchaudio scikit-learn matplotlib pandas

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch: {torch.__version__}")
print(f"Device: {device}")

## 1) Fundamento Matematio: Por que las Compuertas

En una RNN Vanilla, el gradiente retropropagado en el tiempo contiene productos de Jacobianos:

$$\frac{\partial L}{\partial h_t} = \left(\prod_{k=t+1}^{T} J_k\right) \frac{\partial L}{\partial h_T}$$

Si los autovalores dominantes cumplen $|\lambda| < 1$, entonces el producto decae exponencialmente. Resultado: la red olvida informacion lejana.

La idea del **self-loop** (Constant Error Carousel en LSTM) es crear una ruta donde el gradiente pueda fluir con cambios controlados por compuertas, en lugar de multiplicarse por matrices inestables en cada paso.

Diferencia clave de estados:
- **LSTM** separa estado oculto $h_t$ y estado de celda $c_t$.
- **GRU** unifica memoria y estado oculto en un solo vector, con menos compuertas.

In [ ]:
# Demostracion visual del decaimiento exponencial cuando |lambda| < 1
T = 120
lam_values = [0.95, 0.9, 0.8, 0.7]

plt.figure(figsize=(8, 4))
for lam in lam_values:
    grads = [lam**t for t in range(T)]
    plt.plot(grads, label=f"lambda={lam}")

plt.yscale("log")
plt.title("Producto de Jacobianos: decaimiento del gradiente")
plt.xlabel("Pasos temporales")
plt.ylabel("Magnitud relativa (escala log)")
plt.grid(alpha=0.3, linestyle="--")
plt.legend()
plt.show()

## 2) Implementacion de Arquitecturas con torch.nn

### LSTM
`nn.LSTM` devuelve una tupla:

`output, (h_n, c_n)`

donde:
- `h_n`: estado oculto final
- `c_n`: estado de celda final (exclusivo de LSTM)

### GRU
`nn.GRU` devuelve:

`output, h_n`

Punto de comparacion: GRU es mas eficiente computacionalmente porque usa menos compuertas (Update y Reset).

In [ ]:
batch_size, seq_len, input_size = 4, 12, 6
hidden_size = 10
x_demo = torch.randn(batch_size, seq_len, input_size)

lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, batch_first=True)

out_lstm, (h_lstm, c_lstm) = lstm(x_demo)
out_gru, h_gru = gru(x_demo)

print("LSTM")
print(" output shape:", tuple(out_lstm.shape))
print(" h_n shape:", tuple(h_lstm.shape))
print(" c_n shape:", tuple(c_lstm.shape))
print()
print("GRU")
print(" output shape:", tuple(out_gru.shape))
print(" h_n shape:", tuple(h_gru.shape))

## 3) Problema Aplicado I: El Auditor de Reportes Financieros (Many-to-One)

**Objetivo:** comparar RNN Vanilla vs LSTM vs GRU en secuencias largas (`T > 100`).

Simulacion del contexto:
- La pista de fraude aparece en el primer parrafo (tiempo `t=0`).
- La decision se toma al final de la secuencia (`t=T`).

Esto crea una tarea de memoria de largo plazo donde una RNN simple suele degradarse por desvanecimiento de gradiente.

In [ ]:
def make_long_memory_dataset(n_samples=5000, T=120, noise_std=1.0):
    y = np.random.randint(0, 2, size=n_samples).astype(np.int64)
    X = np.random.normal(0, noise_std, size=(n_samples, T, 1)).astype(np.float32)

    # Pista al inicio: +1 si fraude, -1 si no fraude
    X[:, 0, 0] = (2 * y - 1).astype(np.float32)

    # Pequeno ruido extra en la pista para evitar trivialidad
    X[:, 0, 0] += np.random.normal(0, 0.10, size=n_samples).astype(np.float32)
    return X, y

X_mem, y_mem = make_long_memory_dataset(n_samples=6000, T=140)
X_train, X_test, y_train, y_test = train_test_split(
    X_mem, y_mem, test_size=0.25, random_state=SEED, stratify=y_mem
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape: ", X_test.shape, y_test.shape)

In [ ]:
class ManyToOneClassifier(nn.Module):
    def __init__(self, rnn_type="rnn", input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.rnn_type = rnn_type

        if rnn_type == "rnn":
            self.rnn = nn.RNN(input_size, hidden_size, num_layers=num_layers, batch_first=True, nonlinearity="tanh")
        elif rnn_type == "lstm":
            self.rnn = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        elif rnn_type == "gru":
            self.rnn = nn.GRU(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        else:
            raise ValueError("rnn_type debe ser 'rnn', 'lstm' o 'gru'")

        self.head = nn.Linear(hidden_size, 2)

    def forward(self, x):
        out, h = self.rnn(x)

        if self.rnn_type == "lstm":
            h_n = h[0]  # h_n de (h_n, c_n)
        else:
            h_n = h

        last_hidden = h_n[-1]  # ultima capa
        logits = self.head(last_hidden)
        return logits

def batchify(X, y, batch_size=128, shuffle=True):
    idx = np.arange(len(X))
    if shuffle:
        np.random.shuffle(idx)

    for i in range(0, len(X), batch_size):
        j = idx[i:i+batch_size]
        xb = torch.tensor(X[j], dtype=torch.float32, device=device)
        yb = torch.tensor(y[j], dtype=torch.long, device=device)
        yield xb, yb

def train_eval_model(rnn_type, X_train, y_train, X_test, y_test, epochs=10, lr=1e-3, clip=1.0):
    model = ManyToOneClassifier(rnn_type=rnn_type, hidden_size=32).to(device)
    optim = torch.optim.Adam(model.parameters(), lr=lr)

    history = []
    for ep in range(epochs):
        model.train()
        losses = []
        for xb, yb in batchify(X_train, y_train, batch_size=128, shuffle=True):
            optim.zero_grad()
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
            optim.step()
            losses.append(loss.item())

        model.eval()
        all_pred, all_true = [], []
        with torch.no_grad():
            for xb, yb in batchify(X_test, y_test, batch_size=256, shuffle=False):
                logits = model(xb)
                pred = torch.argmax(logits, dim=1)
                all_pred.extend(pred.cpu().numpy().tolist())
                all_true.extend(yb.cpu().numpy().tolist())

        f1 = f1_score(all_true, all_pred)
        history.append((np.mean(losses), f1))
        print(f"[{rnn_type.upper()}] Ep {ep+1:02d} | loss={np.mean(losses):.4f} | F1={f1:.4f}")

    return model, history

In [ ]:
histories = {}
for model_name in ["rnn", "lstm", "gru"]:
    _, hist = train_eval_model(model_name, X_train, y_train, X_test, y_test, epochs=8, lr=2e-3, clip=1.0)
    histories[model_name] = hist

summary = []
for k, hist in histories.items():
    final_loss, final_f1 = hist[-1]
    summary.append({"model": k.upper(), "final_loss": final_loss, "final_f1": final_f1})

df_summary = pd.DataFrame(summary).sort_values("final_f1", ascending=False).reset_index(drop=True)
display(df_summary)

plt.figure(figsize=(10, 4))
for k in histories:
    f1_curve = [x[1] for x in histories[k]]
    plt.plot(f1_curve, marker="o", label=k.upper())

plt.title("F1 por epoca: RNN vs LSTM vs GRU (Many-to-One)")
plt.xlabel("Epoca")
plt.ylabel("F1")
plt.grid(alpha=0.3, linestyle="--")
plt.legend()
plt.show()

### Discusion esperada

En tareas de memoria larga, la RNN Vanilla tiende a acercarse al azar (F1 cercano a 0.5), mientras LSTM y GRU conservan mejor la pista inicial gracias a compuertas.

Si tu RNN mejora mas de lo esperado, aumenta `T` (por ejemplo, 250 o 500) para forzar dependencias mas largas.

## 4) Problema Aplicado II: Pronostico de Demanda Energetica (Seq2Seq)

**Objetivo:** predecir los proximos 7 dias dado historial de 30 dias.

Razones para encoder-decoder:
- El codificador resume historial en un contexto.
- El decodificador genera la secuencia futura paso a paso.
- LSTM/GRU capturan estacionalidad y tendencia mejor que modelos sin memoria.

In [ ]:
def generate_energy_series(n_days=1800):
    t = np.arange(n_days)
    weekly = 1.5 * np.sin(2 * np.pi * t / 7)
    monthly = 0.8 * np.sin(2 * np.pi * t / 30)
    trend = 0.002 * t
    noise = np.random.normal(0, 0.25, size=n_days)
    y = 10 + weekly + monthly + trend + noise
    return y.astype(np.float32)

def make_windows(series, in_len=30, out_len=7):
    X, Y = [], []
    for i in range(len(series) - in_len - out_len):
        X.append(series[i:i+in_len])
        Y.append(series[i+in_len:i+in_len+out_len])
    X = np.array(X, dtype=np.float32)
    Y = np.array(Y, dtype=np.float32)
    return X[..., None], Y[..., None]

series = generate_energy_series(2200)
X_seq, Y_seq = make_windows(series, in_len=30, out_len=7)

split = int(0.8 * len(X_seq))
X_tr, Y_tr = X_seq[:split], Y_seq[:split]
X_te, Y_te = X_seq[split:], Y_seq[split:]

plt.figure(figsize=(10, 3))
plt.plot(series[:200], lw=1.5)
plt.title("Serie sintetica de demanda energetica (muestra)")
plt.grid(alpha=0.3, linestyle="--")
plt.show()

print("Train windows:", X_tr.shape, Y_tr.shape)
print("Test windows:", X_te.shape, Y_te.shape)

In [ ]:
class EncoderGRU(nn.Module):
    def __init__(self, input_size=1, hidden_size=48):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)

    def forward(self, x):
        _, h = self.gru(x)
        return h

class DecoderGRU(nn.Module):
    def __init__(self, input_size=1, hidden_size=48, output_size=1):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x_step, h):
        out, h = self.gru(x_step, h)
        y_step = self.fc(out)
        return y_step, h

class Seq2SeqGRU(nn.Module):
    def __init__(self, hidden_size=48):
        super().__init__()
        self.encoder = EncoderGRU(1, hidden_size)
        self.decoder = DecoderGRU(1, hidden_size, 1)

    def forward(self, x_enc, out_len=7, y_teacher=None, teacher_forcing=0.5):
        h = self.encoder(x_enc)
        batch = x_enc.size(0)

        dec_input = x_enc[:, -1:, :]
        preds = []
        for t in range(out_len):
            y_step, h = self.decoder(dec_input, h)
            preds.append(y_step)

            if (y_teacher is not None) and (random.random() < teacher_forcing):
                dec_input = y_teacher[:, t:t+1, :]
            else:
                dec_input = y_step

        return torch.cat(preds, dim=1)

def batchify_seq(X, Y, batch_size=128, shuffle=True):
    idx = np.arange(len(X))
    if shuffle:
        np.random.shuffle(idx)
    for i in range(0, len(X), batch_size):
        j = idx[i:i+batch_size]
        xb = torch.tensor(X[j], dtype=torch.float32, device=device)
        yb = torch.tensor(Y[j], dtype=torch.float32, device=device)
        yield xb, yb

In [ ]:
model_seq = Seq2SeqGRU(hidden_size=48).to(device)
optim_seq = torch.optim.Adam(model_seq.parameters(), lr=1e-3)

train_losses = []
for ep in range(20):
    model_seq.train()
    batch_losses = []
    for xb, yb in batchify_seq(X_tr, Y_tr, batch_size=128, shuffle=True):
        optim_seq.zero_grad()
        pred = model_seq(xb, out_len=7, y_teacher=yb, teacher_forcing=0.6)
        loss = F.mse_loss(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_seq.parameters(), max_norm=1.0)
        optim_seq.step()
        batch_losses.append(loss.item())

    train_losses.append(np.mean(batch_losses))
    if (ep + 1) % 5 == 0:
        print(f"Ep {ep+1:02d} | loss={train_losses[-1]:.5f}")

plt.figure(figsize=(8, 3))
plt.plot(train_losses, marker="o")
plt.title("Evolucion de perdida (Seq2Seq GRU)")
plt.xlabel("Epoca")
plt.ylabel("MSE")
plt.grid(alpha=0.3, linestyle="--")
plt.show()

model_seq.eval()
with torch.no_grad():
    xb = torch.tensor(X_te[:1], dtype=torch.float32, device=device)
    yb = torch.tensor(Y_te[:1], dtype=torch.float32, device=device)
    pred = model_seq(xb, out_len=7, y_teacher=None, teacher_forcing=0.0)

true_future = yb[0, :, 0].cpu().numpy()
pred_future = pred[0, :, 0].cpu().numpy()

plt.figure(figsize=(8, 3))
plt.plot(range(30), X_te[0, :, 0], label="Historial (30 dias)")
plt.plot(range(30, 37), true_future, marker="o", label="Real (7 dias)")
plt.plot(range(30, 37), pred_future, marker="o", label="Predicho (7 dias)")
plt.title("Pronostico de demanda energetica")
plt.xlabel("Dia relativo")
plt.ylabel("Consumo")
plt.grid(alpha=0.3, linestyle="--")
plt.legend()
plt.show()

## 5) El Tip del Doctor: Optimizacion Estable

### A) Inicializacion de sesgo de olvido en LSTM
Buena practica: inicializar bias de forget gate en 1 para favorecer retencion inicial de memoria.

### B) Gradient Clipping
Aplicar `torch.nn.utils.clip_grad_norm_` evita explosion del gradiente en secuencias largas.

In [ ]:
def init_lstm_forget_bias(lstm_module, value=1.0):
    for name, param in lstm_module.named_parameters():
        if "bias_ih" in name or "bias_hh" in name:
            hidden = param.shape[0] // 4
            # Orden de compuertas en PyTorch LSTM: i, f, g, o
            param.data[hidden:2*hidden].fill_(value)

lstm_probe = nn.LSTM(input_size=1, hidden_size=16, batch_first=True)
b_before = lstm_probe.bias_ih_l0.detach().clone()
init_lstm_forget_bias(lstm_probe, value=1.0)
b_after = lstm_probe.bias_ih_l0.detach().clone()

h = b_after.shape[0] // 4
print("Promedio forget bias antes:", b_before[h:2*h].mean().item())
print("Promedio forget bias despues:", b_after[h:2*h].mean().item())

print("Ejemplo de clipping en entrenamiento:")
print("torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)")

## 6) Ejercicios de Reto

1. **Sensibilidad al largo:** aumenta `T` a 500 en el problema financiero y reporta F1 para RNN, LSTM y GRU.
2. **Ablacion de compuertas:** implementa una celda LSTM manual con forget gate fija en 1 y analiza convergencia.
3. **Comparacion de costo:** mide tiempo por epoca de LSTM vs GRU manteniendo el mismo `hidden_size`.
4. **Generalizacion:** agrega drift de concepto (cambio de distribucion temporal) y evalua robustez.

## Recursos de Apoyo

- Documentacion oficial de `torch.nn.LSTM`: https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
- Documentacion oficial de `torch.nn.GRU`: https://pytorch.org/docs/stable/generated/torch.nn.GRU.html
- Goodfellow, Bengio, Courville (Deep Learning), capitulo de modelos secuenciales.

Sugerencia didactica: despues de correr todo el notebook, pide a los estudiantes escribir una conclusion comparando la interpretabilidad operativa de compuertas en LSTM vs GRU para cada problema ABP.